# Retrain v3 — curated bridge/concrete mix

Trains **your custom U-Net** (SE blocks, bottleneck dropout, deep supervision) on UAV Kaggle + DeepCrack + Auto-ROS-LAB UAV 11k. Run cells top to bottom.

**Before you start (upload to MyDrive):**
- `crack-seg_revision_8-2.zip` — the repo code (get it from the project folder; if you've pushed `revision_8-2` to GitHub, you can skip this and the notebook will clone).
- `kaggle.json` — for the UAV dataset download (kaggle.com → Settings → API → Create New Token). Required unless you re-upload `dataset_split.zip` to `MyDrive/bridge_crack_detection/` to keep the original 220/47/48 split.
- DeepCrack's license is non-commercial research/educational — fine for this project.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/crack_v3'
DATA_ROOT = '/content/crack_data'
OUT = DATA_ROOT + '/dataset_split'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print('Drive root:', DRIVE_ROOT)
print('Data out :', OUT)

## 2. Get the code + dependencies

### (Optional) Where am I? — prints every path the notebook uses

In [ ]:
import os
print('CWD          :', os.getcwd())
print('MyDrive items:', sorted(os.listdir('/content/drive/MyDrive')))
print()
print('Notebook variables in this session:')
for v in ('REPO', 'DRIVE_ROOT', 'DATA_ROOT', 'OUT'):
    print(f'  {v:12s} = {globals().get(v)}')
print()
print('Expected paths:')
for label, p in [
    ('repo zip        ', '/content/drive/MyDrive/crack-seg_revision_8-2.zip'),
    ('repo folder     ', '/content/drive/MyDrive/crack-seg'),
    ('dataset_split   ', '/content/drive/MyDrive/bridge_crack_detection/dataset_split.zip'),
    ('kaggle.json     ', '/content/drive/MyDrive/kaggle.json'),
    ('DRIVE_ROOT (out)', '/content/drive/MyDrive/crack_v3'),
    ('DATA_ROOT       ', '/content/crack_data'),
    ('OUT             ', '/content/crack_data/dataset_split'),
]:
    print(f'  {label} {p}  ->  exists: {os.path.exists(p)}')

In [ ]:
import os, sys, shutil, glob

def find_repo():
    if os.path.isdir('/content/crack-seg'):
        return '/content/crack-seg'
    if os.path.isdir('/content/src') and os.path.isdir('/content/scripts'):
        return '/content'
    return None

REPO = find_repo()
if REPO is None:
    if os.path.isdir('/content/drive/MyDrive/crack-seg'):
        REPO = '/content/crack-seg'
        shutil.copytree('/content/drive/MyDrive/crack-seg', REPO)
        print('Copied repo from Drive folder.')
    else:
        zips = sorted(glob.glob('/content/drive/MyDrive/crack*seg*.zip'))
        zips += sorted(glob.glob('/content/drive/MyDrive/**/crack*seg*.zip', recursive=True))
        if zips:
            shutil.unpack_archive(zips[0], '/content')
            print('Unzipped repo from Drive:', zips[0])
        else:
            print('No repo on Drive yet. Pick crack-seg_revision_8-2.zip in the file dialog that just opened.')
            from google.colab import files
            uploaded = files.upload()
            for name in uploaded:
                if name.endswith('.zip'):
                    shutil.unpack_archive(name, '/content')
                    break
        REPO = find_repo() or '/content/crack-seg'
    if not os.path.isdir(REPO) and not (os.path.isdir('/content/src') and os.path.isdir('/content/scripts')):
        ret = os.system('git clone -b revision_8-2 https://github.com/Ishaan1402/crack-seg.git /content/crack-seg')
        if ret != 0 or not os.path.isdir('/content/crack-seg'):
            print('\nCould not get the repo code. Do one of:')
            print('  1) upload crack-seg_revision_8-2.zip to MyDrive and rerun this cell, or')
            print('  2) click the file dialog when this cell runs and pick the zip, or')
            print('  3) push revision_8-2 to GitHub and rerun this cell.')
            raise SystemExit(1)
        REPO = '/content/crack-seg'
os.chdir(REPO)
sys.path.insert(0, REPO)
# Minimal install ONLY: Colab already ships torch/numpy/opencv/pydantic.
# Installing requirements.txt would downgrade them and break the runtime.
os.system('pip install -q albumentations kagglehub gdown')
print('Setup complete in', os.getcwd())

### Kaggle credentials (required for the UAV source)

Provide them via **one** of: (1) pasting your username + API key into the cell below, (2) `MyDrive/kaggle.json`, or (3) Colab Secrets (`KAGGLE_USERNAME` + `KAGGLE_KEY`).

Find your credentials at kaggle.com → avatar → **Settings** → **API** → **Create New Token** (the current UI shows the username and key in-page — copy them).

In [ ]:
import json, os
from scripts import colab_data as cd

def _write_kaggle(user, key):
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
        json.dump({'username': user, 'key': key}, f)
    os.environ['KAGGLE_USERNAME'] = str(user)
    os.environ['KAGGLE_KEY'] = str(key)

ok = cd.setup_kaggle_credentials('/content/drive/MyDrive/kaggle.json')
if ok:
    print('Kaggle credentials loaded from MyDrive/kaggle.json.')
else:
    try:
        from google.colab import userdata
        _write_kaggle(userdata.get('KAGGLE_USERNAME'), userdata.get('KAGGLE_KEY'))
        ok = True
        print('Kaggle credentials loaded from Colab Secrets.')
    except Exception:
        pass
if not ok:
    print('No kaggle.json in MyDrive and no Colab Secrets found.')
    print('Paste your username + API key into the next cell, or re-upload dataset_split.zip to')
    print('MyDrive/bridge_crack_detection/ to skip Kaggle entirely.')

In [ ]:
#@title Paste your Kaggle credentials here (kaggle.com -> Settings -> API)
KAGGLE_USERNAME = "your_kaggle_username" #@param {type:"string"}
KAGGLE_KEY = "paste_your_api_key_here" #@param {type:"string"}

import json, os
if KAGGLE_USERNAME.startswith('your_') or KAGGLE_KEY.startswith('paste_'):
    print('Replace the two placeholder values above with your real credentials, then rerun this cell.')
else:
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
        json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
    os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
    os.environ['KAGGLE_KEY'] = KAGGLE_KEY
    print('Kaggle credentials set from the form.')

## 3. Download & stage sources

### UAV Kaggle (primary — fresh 70/15/15 split via Kaggle, or original split if dataset_split.zip is on Drive)

In [ ]:
from scripts import colab_data as cd
DRIVE_ZIP = '/content/drive/MyDrive/bridge_crack_detection/dataset_split.zip'
mode = cd.download_uav(DRIVE_ZIP if os.path.exists(DRIVE_ZIP) else None, DATA_ROOT, OUT)
print('UAV staged from:', mode, '(drive = original 220/47/48 split, kaggle = fresh 70/15/15)')

### DeepCrack (train only; its test set is held out for evaluation)

In [ ]:
dc_train_img, dc_train_lab, DC_TEST_IMG, DC_TEST_LAB = cd.download_deepcrack(DATA_ROOT)
cd.stage(dc_train_img, dc_train_lab, OUT, source='deepcrack', cap=300, resize=512, val_frac=0.1, test_frac=0.0, seed=42)
print('DeepCrack test held out for eval:', DC_TEST_IMG)
print('                                 ', DC_TEST_LAB)

### Big volume source (Auto-ROS-LAB UAV 11k, with merged 11.2k fallback)

Tries the UAV 11k Drive file first (closest to your use case). When Google Drive rate-limits it, the cell automatically falls back to the merged 11.2k dataset (12 public crack datasets, 448×448, images+masks). Set `SKIP_BIG_SOURCE = True` to skip both and train on UAV Kaggle + DeepCrack only.

In [ ]:
import glob, os, gdown, zipfile
SKIP_BIG_SOURCE = False  # set True to skip both big sources

def _find_im_msk(root):
    def pick(kinds):
        cands = []
        for d in glob.glob(root + '/**/*', recursive=True):
            if os.path.isdir(d) and any(t in os.path.basename(d).lower() for t in kinds):
                n = len(os.listdir(d))
                if n > 10:
                    cands.append((n, d))
        top = [d for n, d in cands if os.path.dirname(d) == root]
        if top:
            return top[0]
        return sorted(cands, reverse=True)[0][1]
    return pick(('image', 'img')), pick(('mask', 'label', 'lab', 'gt'))

if not SKIP_BIG_SOURCE:
    staged = False
    try:
        imgs, msks = cd.download_uav11k(DATA_ROOT)
        cd.stage(imgs, msks, OUT, source='uav11k', cap=8000, resize=512, val_frac=0.1, test_frac=0.05, seed=42)
        staged = True
        print('UAV 11k staged (cap=8000).')
    except Exception as exc:
        print('UAV 11k failed (' + str(exc)[:100] + ')')
        print('Falling back to the merged 11.2k dataset...')
    if not staged:
        ZIP = DATA_ROOT + '/crack11k.zip'
        RAW = DATA_ROOT + '/crack11k_raw'
        try:
            if not os.path.exists(ZIP):
                gdown.download(id='1xrOqv0-3uMHjZyEUrerOYiYXW_E8SUMP', output=ZIP, quiet=False, fuzzy=True)
            if not (os.path.isdir(RAW) and any(glob.glob(RAW + '/**/*', recursive=True))):
                os.makedirs(RAW, exist_ok=True)
                with zipfile.ZipFile(ZIP) as z:
                    z.extractall(RAW)
            imgs, msks = _find_im_msk(RAW)
            print('Using images:', imgs, '| masks:', msks)
            cd.stage(imgs, msks, OUT, source='merged11k', cap=8000, resize=512, val_frac=0.1, test_frac=0.05, seed=42)
            print('Merged 11.2k staged (cap=8000).')
        except Exception as exc2:
            print('Merged 11.2k failed too:', exc2)
            print('Set SKIP_BIG_SOURCE = True to continue with UAV Kaggle + DeepCrack only.')

## 4. Staging summary (per-source sanity check)

In [ ]:
import json
with open(OUT + '/manifest.json') as f:
    manifest = json.load(f)
print(f"{'source':10s} {'train':>7s} {'val':>7s} {'test':>7s} {'crack%':>8s}")
for src, m in manifest.items():
    print(f"{src:10s} {m['train']:7d} {m['val']:7d} {m['test']:7d} {m['mean_crack_ratio']*100:7.2f}%")

## 5. Train v3 (your U-Net, upgraded)

- Primary run: narrow `[32,64,128,256]` + SE + dropout 0.1 + deep supervision, 512px, strong aug, AMP, lr 5e-4.
- If val Dice stays near 0 after a few epochs (the model predicts no cracks), stop and rerun with `--lr 1e-4` (and optionally `--bce-weight 0.7`).
- Val metrics and the best checkpoint are the **raw model**; a final EMA-smoothed snapshot is also saved as `<out>.ema.pth`.
- Optional wide comparison: set `RUN_WIDE = True` and rerun this cell.
- Hardware presets auto-select batch size (T4/L4 vs A100/V100).
- If the session dies, rerun with `--checkpoint` pointing at the last `.pth` to resume from those weights.

### Backup / restore staged data (use before a runtime factory reset)

If you ever need to factory-reset the runtime (e.g. after a broken pip install), run the backup cell to copy the downloaded + staged data to Drive, then after reset + Drive mount, run the restore line before the download cells so nothing is re-downloaded.

In [ ]:
import os, shutil
# BACKUP (run before factory reset):
shutil.copytree(DATA_ROOT, DRIVE_ROOT + '/crack_data_backup', dirs_exist_ok=True)
print('Backed up staged data to', DRIVE_ROOT + '/crack_data_backup')

# RESTORE (run after reset, once Drive is mounted and DATA_ROOT is set):
# shutil.copytree(DRIVE_ROOT + '/crack_data_backup', DATA_ROOT, dirs_exist_ok=True)
# print('Restored staged data from Drive.')

In [ ]:
from scripts import train as train_mod
RUN_WIDE = False
features = '64,128,256,512' if RUN_WIDE else '32,64,128,256'
out_name = 'unet_v3_wide.pth' if RUN_WIDE else 'unet_v3_narrow.pth'
out_path = DATA_ROOT + '/' + out_name
train_mod.main([
    '--data-dir', OUT,
    '--out', out_path,
    '--epochs', '30',
    '--lr', '5e-4',
    '--resize', '512',
    '--features', features,
    '--se', '--dropout', '0.1', '--deep-supervision',
    '--aug', 'strong',
    '--amp',
])

## 6. Save to Drive + evaluation commands

In [ ]:
import shutil
shutil.copy(out_path, DRIVE_ROOT + '/' + out_name)
print('Saved to Drive:', DRIVE_ROOT + '/' + out_name)
print()
print('HF upload (after huggingface-cli login):')
print(f'  huggingface-cli upload ishaan1402/crack-seg {out_path} unet_v3.pth')
dc_test = globals().get('DC_TEST_IMG')
dc_lab = globals().get('DC_TEST_LAB')
if dc_test and dc_lab and os.path.isdir(dc_test):
    print()
    print('Cross-domain eval on DeepCrack test:')
    print(f'  PYTHONPATH={REPO} python scripts/verify_metrics.py --checkpoint {out_path} --images {dc_test} --masks {dc_lab} --mode both --thresholds 0.3 0.4 0.5 0.6 0.7')
else:
    print()
    print('DeepCrack test not downloaded in this session — run the DeepCrack cell, then the "7. Actual test run" cell for cross-domain eval.')
print()
print('In-distribution eval on the staged test split (all sources, untouched during training):')
print(f'  PYTHONPATH={REPO} python scripts/verify_metrics.py --checkpoint {out_path} --images {OUT}/test/images --masks {OUT}/test/masks --mode both --thresholds 0.5')
print('  (files named uav_* in that split are the UAV Kaggle subset)')

## 7. Actual test run (same protocol for every model)

Runs the real evaluation now: the **staged test split** (untouched during training) and the **DeepCrack test** (never trained on). The v3 checkpoint from this session is compared against the two existing models fetched from `ishaan1402/crack-seg` on Hugging Face. Protocol: direct full-image inference, threshold 0.5, global + per-image Dice.

In [ ]:
import os
import numpy as np
import torch
import cv2
from src.models.checkpoint import load_unet_checkpoint

THRESH = 0.5
out_path = globals().get('out_path')
RUN_WIDE = globals().get('RUN_WIDE', False)

MODELS_DIR = DATA_ROOT + '/models'
os.makedirs(MODELS_DIR, exist_ok=True)
candidates = []

# 1) the v3 checkpoint trained in this session
if out_path and os.path.exists(out_path):
    candidates.append(('v3-' + ('wide' if RUN_WIDE else 'narrow'), out_path))

# 2) the two existing models from Hugging Face (public, no auth)
try:
    from huggingface_hub import hf_hub_download
    n2 = MODELS_DIR + '/unet_narrow_v2.pth'
    if not os.path.exists(n2):
        hf_hub_download(repo_id='ishaan1402/crack-seg', filename='best_unet.pth', local_dir=MODELS_DIR)
        os.rename(MODELS_DIR + '/best_unet.pth', n2)
    candidates.append(('narrow-v2 (published)', n2))
    w1 = MODELS_DIR + '/unet_wide_v1.pth'
    if not os.path.exists(w1):
        hf_hub_download(repo_id='ishaan1402/crack-seg', filename='unet_wide_v1.pth', local_dir=MODELS_DIR)
    if os.path.exists(w1):
        candidates.append(('wide-v1 (original)', w1))
except Exception as exc:
    print('Could not fetch existing models from HF:', exc)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def predict(model, rgb):
    x = torch.from_numpy(((rgb.astype(np.float32) / 255.0 - MEAN) / STD).transpose(2, 0, 1)[None]).to(device)
    with torch.inference_mode():
        return torch.sigmoid(model(x)).squeeze().cpu().numpy()

def run_set(name, img_dir, msk_dir):
    if not (os.path.isdir(img_dir) and os.path.isdir(msk_dir)):
        print(f'SKIP {name}: missing {img_dir} or {msk_dir}')
        return
    imgs, msks = sorted(os.listdir(img_dir)), sorted(os.listdir(msk_dir))
    print(f'\n=== {name} ({len(imgs)} images, threshold={THRESH}) ===')
    print(f"{'model':24s} {'Dice':>6s} {'IoU':>6s} {'Rec':>6s} {'Prec':>6s}")
    for label, ckpt in candidates:
        model, _ = load_unet_checkpoint(ckpt, device)
        g = {'tp': 0, 'fp': 0, 'fn': 0}
        per = []
        for ip, mp in zip(imgs, msks):
            rgb = cv2.cvtColor(cv2.imread(os.path.join(img_dir, ip)), cv2.COLOR_BGR2RGB)
            gt = cv2.imread(os.path.join(msk_dir, mp), cv2.IMREAD_GRAYSCALE) > 127
            pred = predict(model, rgb) > THRESH
            tp = int(np.sum(pred & gt)); fp = int(np.sum(pred & ~gt)); fn = int(np.sum(~pred & gt))
            g['tp'] += tp; g['fp'] += fp; g['fn'] += fn
            per.append((2 * tp + 1e-6) / (2 * tp + fp + fn + 1e-6))
        eps = 1e-6
        dice = (2 * g['tp'] + eps) / (2 * g['tp'] + g['fp'] + g['fn'] + eps)
        iou = (g['tp'] + eps) / (g['tp'] + g['fp'] + g['fn'] + eps)
        rec = (g['tp'] + eps) / (g['tp'] + g['fn'] + eps)
        prec = (g['tp'] + eps) / (g['tp'] + g['fp'] + eps)
        print(f"{label:24s} {dice:6.4f} {iou:6.4f} {rec:6.4f} {prec:6.4f}")
        print(f"{'':24s} macro Dice: {float(np.mean(per)):.4f}")

if not candidates:
    print('No checkpoints to evaluate (train first, or the HF download failed).')
else:
    run_set('staged test split', OUT + '/test/images', OUT + '/test/masks')
    dc_test = globals().get('DC_TEST_IMG')
    if dc_test:
        run_set('deepcrack test', dc_test, globals().get('DC_TEST_LAB'))

## 8. Quick inference visuals (for README / ishaan1402.github.io)

Runs the trained v3 model on a few test images and saves README-style panels (original | mask | heatmap) plus a ground-truth comparison. Panels go to `/content/crack_data/visuals/`, Drive, and are shown inline.

In [ ]:
import os, glob, shutil
import numpy as np, cv2, torch
import matplotlib.pyplot as plt

from src.models.checkpoint import load_unet_checkpoint
from src.config.schema import SystemSettings
from src.inference.sliding_window import SlidingWindowPredictor
from src.utils.visualizations import create_visual_overlay

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
settings = SystemSettings.load_from_yaml('config/config.yaml')
model, _ = load_unet_checkpoint(out_path, device)
predictor = SlidingWindowPredictor(model, settings, device)

VIS = DATA_ROOT + '/visuals'
os.makedirs(VIS, exist_ok=True)

test_imgs = sorted(glob.glob(OUT + '/test/images/uav_*.jpg'))[:2]
test_imgs += sorted(glob.glob(OUT + '/test/images/merged11k_*.jpg'))[:2]
example = REPO + '/input/example_1.jpeg'
if os.path.exists(example):
    test_imgs.append(example)
if not test_imgs:
    raise SystemExit('No test images found — run the staging cells first.')

# Optional: upload your own photos instead
# from google.colab import files
# uploaded = files.upload()
# test_imgs += [n for n in uploaded if n.lower().endswith(('.jpg', '.jpeg', '.png'))]

rows = []
for p in test_imgs:
    bgr = cv2.imread(p)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    _, mask, car = predictor.predict_large_image(rgb, threshold=0.5)
    panel = create_visual_overlay(bgr, mask, 'both', settings)
    stem = os.path.splitext(os.path.basename(p))[0]
    out = f'{VIS}/{stem}_panel.jpg'
    cv2.imwrite(out, panel)
    rows.append((stem, panel, car))
    print(f'{stem}: crack-area-ratio={car:.4f} -> {out}')

pair_stem = os.path.splitext(os.path.basename(test_imgs[0]))[0]
pair_msk = OUT + '/test/masks/' + pair_stem + '.png'
if os.path.exists(pair_msk):
    bgr = cv2.imread(test_imgs[0])
    gt = cv2.imread(pair_msk, cv2.IMREAD_GRAYSCALE)
    _, pair_mask, _ = predictor.predict_large_image(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), threshold=0.5)
    gt_rgb = cv2.cvtColor(gt, cv2.COLOR_GRAY2BGR)
    pred_rgb = cv2.cvtColor((pair_mask * 255).astype(np.uint8), cv2.COLOR_GRAY2BGR)
    gt_grid = np.hstack((bgr, gt_rgb, pred_rgb))
    cv2.imwrite(f'{VIS}/{pair_stem}_gt_comparison.jpg', gt_grid)
    print('GT comparison saved:', f'{VIS}/{pair_stem}_gt_comparison.jpg')

fig, axes = plt.subplots(len(rows), 1, figsize=(16, 5.5 * len(rows)))
if len(rows) == 1:
    axes = [axes]
for ax, (stem, panel, car) in zip(axes, rows):
    ax.imshow(cv2.cvtColor(panel, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{stem} | crack-area: {car:.2%}', fontsize=11)
    ax.axis('off')
plt.tight_layout()
grid = f'{VIS}/grid.png'
plt.savefig(grid, dpi=130)
plt.show()

shutil.copytree(VIS, DRIVE_ROOT + '/visuals', dirs_exist_ok=True)
print('Saved to', VIS)
print('Copied to Drive:', DRIVE_ROOT + '/visuals')

## 9. Per-image IoU/Dice on the staged test split

Computes per-image Dice/IoU against ground truth for the staged test set, prints summary stats plus best/worst examples, and saves original | GT | prediction panels for the top/bottom images.

In [ ]:
import os
import numpy as np, cv2, torch
import matplotlib.pyplot as plt

from src.models.checkpoint import load_unet_checkpoint

THRESH = 0.5
N_LIMIT = 0      # 0 = all staged test images; set e.g. 100 to cap
N_SHOW = 3       # best/worst examples to rank and save panels for

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, _ = load_unet_checkpoint(out_path, device)
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def predict(model, rgb):
    x = torch.from_numpy(((rgb.astype(np.float32) / 255.0 - MEAN) / STD).transpose(2, 0, 1)[None]).to(device)
    with torch.inference_mode():
        return torch.sigmoid(model(x)).squeeze().cpu().numpy()

img_dir = OUT + '/test/images'
msk_dir = OUT + '/test/masks'
mask_map = {os.path.splitext(n)[0]: os.path.join(msk_dir, n) for n in os.listdir(msk_dir)}
results = []
empty_gt = 0
for name in sorted(os.listdir(img_dir)):
    stem = os.path.splitext(name)[0]
    if stem not in mask_map:
        continue
    if N_LIMIT and len(results) >= N_LIMIT:
        break
    rgb = cv2.cvtColor(cv2.imread(os.path.join(img_dir, name)), cv2.COLOR_BGR2RGB)
    gt = cv2.imread(mask_map[stem], cv2.IMREAD_GRAYSCALE) > 127
    if not gt.any():
        empty_gt += 1
        continue
    pred = predict(model, rgb) > THRESH
    tp = int(np.sum(pred & gt)); fp = int(np.sum(pred & ~gt)); fn = int(np.sum(~pred & gt))
    eps = 1e-6
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    results.append((stem, dice, iou, tp, fp, fn))

results.sort(key=lambda r: r[1])
dice_all = np.array([r[1] for r in results])
iou_all = np.array([r[2] for r in results])
print(f'Images with cracks: {len(results)} | empty-GT skipped: {empty_gt}')
print(f'Dice mean={dice_all.mean():.4f} median={np.median(dice_all):.4f} p10={np.percentile(dice_all, 10):.4f} p90={np.percentile(dice_all, 90):.4f}')
print(f'IoU  mean={iou_all.mean():.4f} median={np.median(iou_all):.4f} p10={np.percentile(iou_all, 10):.4f} p90={np.percentile(iou_all, 90):.4f}')

print(f'\nWorst {N_SHOW}:')
for stem, dice, iou, tp, fp, fn in results[:N_SHOW]:
    print(f'  {stem:34s} dice={dice:.4f} iou={iou:.4f} tp={tp} fp={fp} fn={fn}')
print(f'Best {N_SHOW}:')
for stem, dice, iou, tp, fp, fn in list(reversed(results[-N_SHOW:])):
    print(f'  {stem:34s} dice={dice:.4f} iou={iou:.4f} tp={tp} fp={fp} fn={fn}')

plt.figure(figsize=(8, 4))
plt.hist(dice_all, bins=25, color='purple', alpha=0.7)
plt.xlabel('per-image Dice'); plt.ylabel('images'); plt.title('Dice distribution (staged test split)')
plt.show()

PER = DATA_ROOT + '/visuals/per_image'
os.makedirs(PER, exist_ok=True)
chosen = [('worst', r) for r in results[:N_SHOW]] + [('best', r) for r in list(reversed(results[-N_SHOW:]))]
fig, axes = plt.subplots(len(chosen), 1, figsize=(15, 4.5 * len(chosen)))
if len(chosen) == 1:
    axes = [axes]
for ax, (tag, (stem, dice, iou, tp, fp, fn)) in zip(axes, chosen):
    bgr = cv2.imread(os.path.join(img_dir, stem + '.jpg'))
    gt = cv2.imread(mask_map[stem], cv2.IMREAD_GRAYSCALE)
    pred = (predict(model, cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)) > THRESH).astype(np.uint8) * 255
    gt_rgb = cv2.cvtColor(gt, cv2.COLOR_GRAY2BGR)
    pred_rgb = cv2.cvtColor(pred, cv2.COLOR_GRAY2BGR)
    panel = np.hstack((bgr, gt_rgb, pred_rgb))
    out = f'{PER}/{tag}_{stem}.jpg'
    cv2.imwrite(out, panel)
    ax.imshow(cv2.cvtColor(panel, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{tag} {stem} | dice={dice:.3f} iou={iou:.3f}', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'{PER}/best_worst_grid.png', dpi=130)
plt.show()
print('Panels saved to', PER)

## 10. Diagnose dice≈0 cases (source pattern check)

Groups per-image results by source, lists every near-zero Dice case with ground-truth stats (inverted? blank? model predicted nothing?), and saves original | GT | prediction panels for them.

In [ ]:
import os, collections
import numpy as np, cv2, torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, _ = load_unet_checkpoint(out_path, device)
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def predict(model, rgb):
    x = torch.from_numpy(((rgb.astype(np.float32) / 255.0 - MEAN) / STD).transpose(2, 0, 1)[None]).to(device)
    with torch.inference_mode():
        return torch.sigmoid(model(x)).squeeze().cpu().numpy()

def source_of(stem):
    parts = stem.split('_')
    return parts[1] if parts and parts[0] == 'merged11k' and len(parts) > 1 else (parts[0] if parts else stem)

img_dir = OUT + '/test/images'
msk_dir = OUT + '/test/masks'
mask_map = {os.path.splitext(n)[0]: os.path.join(msk_dir, n) for n in os.listdir(msk_dir)}
stats = []
for name in sorted(os.listdir(img_dir)):
    stem = os.path.splitext(name)[0]
    if stem not in mask_map:
        continue
    rgb = cv2.cvtColor(cv2.imread(os.path.join(img_dir, name)), cv2.COLOR_BGR2RGB)
    gt = cv2.imread(mask_map[stem], cv2.IMREAD_GRAYSCALE) > 127
    if not gt.any():
        continue
    pred = predict(model, rgb) > 0.5
    tp = int(np.sum(pred & gt)); fp = int(np.sum(pred & ~gt)); fn = int(np.sum(~pred & gt))
    eps = 1e-6
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    stats.append((stem, source_of(stem), dice, float(gt.mean()), bool(pred.any()), tp, fp, fn))

by_src = collections.defaultdict(list)
for s in stats:
    by_src[s[1]].append(s)
print(f"{'source':12s} {'n':>5s} {'meanDice':>8s} {'zeros':>6s}")
for src, rows in sorted(by_src.items()):
    dices = [r[2] for r in rows]
    zeros = sum(1 for d in dices if d < 0.001)
    print(f'{src:12s} {len(rows):5d} {np.mean(dices):8.4f} {zeros:6d}')

zeros = [s for s in stats if s[2] < 0.001]
print(f'\nNear-zero cases: {len(zeros)}')
for stem, src, dice, gt_mean, pred_any, tp, fp, fn in zeros:
    flags = []
    if gt_mean > 0.9:
        flags.append('GT almost all-white (inverted/mismatch?)')
    elif gt_mean > 0.5:
        flags.append(f'GT mostly-white ({gt_mean:.2f})')
    if not pred_any:
        flags.append('model predicted NOTHING')
    print(f'  {stem:38s} src={src:10s} gt_mean={gt_mean:.4f} pred_any={pred_any} flags={flags}')

# Panels for the zero cases so you can eyeball whether GT matches the image
PER = DATA_ROOT + '/visuals/zeros'
os.makedirs(PER, exist_ok=True)
for stem, src, dice, gt_mean, pred_any, tp, fp, fn in zeros[:8]:
    bgr = cv2.imread(os.path.join(img_dir, stem + '.jpg'))
    gt = cv2.imread(mask_map[stem], cv2.IMREAD_GRAYSCALE)
    pred = (predict(model, cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)) > 0.5).astype(np.uint8) * 255
    gt_rgb = cv2.cvtColor(gt, cv2.COLOR_GRAY2BGR)
    pred_rgb = cv2.cvtColor(pred, cv2.COLOR_GRAY2BGR)
    cv2.imwrite(f'{PER}/{stem}.jpg', np.hstack((bgr, gt_rgb, pred_rgb)))
print('Panels saved to', PER)

## 11. Flag likely misaligned pairs (no retrain, no auto-fix)

Scans the staged test split for the mismatch signature (model confidently predicts cracks, GT says ~nothing there), prints the review list, and reports per-source metrics with and without flagged pairs.

In [ ]:
from scripts.flag_bad_pairs import main as flag_main
flag_main([
    '--checkpoint', out_path,
    '--images', OUT + '/test/images',
    '--masks', OUT + '/test/masks',
    '--panels', DATA_ROOT + '/visuals/flagged',
])

## 12. Single-file visuals (examples + best 3)

Saves singular crack / prediction (and ground-truth for best 3) images into `visuals/singles/` — no concatenated panels.

In [ ]:
import os, glob
import numpy as np, cv2, torch

from src.models.checkpoint import load_unet_checkpoint
from src.utils.visualizations import create_visual_overlay
from src.config.schema import SystemSettings

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
settings = SystemSettings.load_from_yaml('config/config.yaml')
model, _ = load_unet_checkpoint(out_path, device)
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def predict(rgb):
    x = torch.from_numpy(((rgb.astype(np.float32) / 255.0 - MEAN) / STD).transpose(2, 0, 1)[None]).to(device)
    with torch.inference_mode():
        return torch.sigmoid(model(x)).squeeze().cpu().numpy()

CRACK_STYLE = 'overlay'  # 'overlay' = green highlight on photo; 'mask' = binary white-on-black

def predict_mask(rgb):
    return (predict(rgb) > 0.5).astype(np.uint8)

def crack_view(bgr, mask):
    if CRACK_STYLE == 'overlay':
        return create_visual_overlay(bgr, mask, 'mask', settings)
    return cv2.cvtColor((mask * 255).astype(np.uint8), cv2.COLOR_GRAY2BGR)

def save_mask(path, mask):
    cv2.imwrite(path, cv2.cvtColor((mask * 255).astype(np.uint8), cv2.COLOR_GRAY2BGR))

EXAMPLES = []
example = REPO + '/input/example_1.jpeg'
if os.path.exists(example):
    EXAMPLES.append(example)
EXAMPLES += sorted(glob.glob(OUT + '/test/images/uav_*.jpg'))[:2]
EXAMPLES += sorted(glob.glob(OUT + '/test/images/merged11k_*.jpg'))[:2]

OUT_EX = DATA_ROOT + '/visuals/singles/examples'
os.makedirs(OUT_EX, exist_ok=True)
for p in EXAMPLES:
    stem = os.path.splitext(os.path.basename(p))[0]
    bgr = cv2.imread(p)
    mask = predict_mask(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    cv2.imwrite(f'{OUT_EX}/{stem}_crack.jpg', crack_view(bgr, mask))
    save_mask(f'{OUT_EX}/{stem}_prediction.jpg', mask)
    print('saved', stem)

img_dir = OUT + '/test/images'
msk_dir = OUT + '/test/masks'
mask_map = {os.path.splitext(n)[0]: os.path.join(msk_dir, n) for n in os.listdir(msk_dir)}
results = []
for name in sorted(os.listdir(img_dir)):
    stem = os.path.splitext(name)[0]
    if stem not in mask_map:
        continue
    rgb = cv2.cvtColor(cv2.imread(os.path.join(img_dir, name)), cv2.COLOR_BGR2RGB)
    gt = cv2.imread(mask_map[stem], cv2.IMREAD_GRAYSCALE) > 127
    if not gt.any():
        continue
    pred = predict(rgb) > 0.5
    tp = int(np.sum(pred & gt)); fp = int(np.sum(pred & ~gt)); fn = int(np.sum(~pred & gt))
    eps = 1e-6
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    results.append((stem, dice))
results.sort(key=lambda r: -r[1])

OUT_B3 = DATA_ROOT + '/visuals/singles/best3'
os.makedirs(OUT_B3, exist_ok=True)
for i, (stem, dice) in enumerate(results[:3], 1):
    bgr = cv2.imread(os.path.join(img_dir, stem + '.jpg'))
    mask = predict_mask(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    gt = cv2.imread(mask_map[stem], cv2.IMREAD_GRAYSCALE)
    cv2.imwrite(f'{OUT_B3}/best{i}_{stem}_crack.jpg', crack_view(bgr, mask))
    save_mask(f'{OUT_B3}/best{i}_{stem}_prediction.jpg', mask)
    cv2.imwrite(f'{OUT_B3}/best{i}_{stem}_ground_truth.jpg', gt)
    print(f'best{i}: {stem} dice={dice:.4f}')

print('Examples ->', OUT_EX)
print('Best 3   ->', OUT_B3)

## Notes

- The staged dataset lives in Colab's ephemeral disk; only checkpoints are copied to Drive. Re-running the download cells after a session reset is expected.
- The UAV source now gets a fresh 70/15/15 split (your old dataset_split is gone). Treat that split as the fixed UAV test set going forward and do not regenerate it between runs.
- `manifest.json` records per-source counts + mean crack ratio — a source with a suspiciously low/high crack% is a red flag worth inspecting before trusting the run.
- After training, update the HF model card with test-set numbers only (run `verify_metrics.py` on the real UAV test split, not just validation).